In [16]:
#apply StandardScaler
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch
from sklearn.utils import resample
#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

# Drop target and ID column & target column
X = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape :", X.shape)

Features shape : (96000, 7)


In [2]:
#Apply StandardScaler
#Now X_scaled contains all features standardized (mean = 0, std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features shape (scaled version):", X_scaled.shape)

Features shape (scaled version): (96000, 7)


In [3]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [10]:
#K-Means on Scaled Data
start_time = time.time()
kmean_scaled = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    kmean_scaled.append({"algorithm": "KMeans", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 1192.6705331802368 seconds
K-Means runtime: 1192.6705 seconds


In [11]:
#Gaussian Mixture (GMM)on Scaled Data
start_time = time.time()
gmm_scaled = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    gmm_scaled.append({"algorithm": "GMM", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 1705.9562449455261 seconds
GMM runtime: 1705.9562 seconds


In [21]:
#Agglomerative Clustering on Scaled + PCA Data
# X_small = resample(X_scaled,  n_samples=5000, random_state=42)  #replace=False,
# print("Features shape :", X_small.shape)
subsample = np.random.choice(len(X_scaled), 10000, replace=False)
X_sub = X_scaled[subsample]
start_time = time.time()
agg_scaled = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg.fit(X_sub)
    labels = agg.labels_
    sil, db, ch = compute_metrics(X_sub, labels)
    agg_scaled.append({"algorithm":"Agglomerative","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 44.161720275878906 seconds
Agglomerative runtime: 44.1617 seconds


In [19]:
#Spectral Clustering on Scaled Data
start_time = time.time()
subsample = np.random.choice(len(X_scaled), 10000, replace=False)
X_sub = X_scaled[subsample]
spec_scaled = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_sub)
    sil, db, ch = compute_metrics(X_sub, labels)
    spec_scaled.append({"algorithm": "Spectral", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 199.35723495483398 seconds
Spectral runtime: 199.3572 seconds


In [17]:
import numpy as np

print("Shape:", X_scaled.shape)
print("NaN:", np.isnan(X_scaled).sum())
print("Inf:", np.isinf(X_scaled).sum())
print("Dtype:", X_scaled.dtype)
print("Max:", np.max(X_scaled))
print("Min:", np.min(X_scaled))

Shape: (96000, 7)
NaN: 0
Inf: 0
Dtype: float32
Max: 4.2852187
Min: -4.561943


In [19]:
#DBSCAN on Scaled Data
start_time = time.time()
dbscan_scaled = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_scaled)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_scaled[mask], labels[mask])
        dbscan_scaled.append({"algorithm": "DBSCAN", "preprocessing": "Scaled", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Dbscan runtime: {runtime:.4f} seconds")

Runtime: 360.57462787628174 seconds
Dbscan runtime: 360.5746 seconds


In [22]:
#BIRCH on Scaled Data
start_time = time.time()
# X_small = resample(X_scaled,  n_samples=5000, random_state=42)  #replace=False,
# print("Features shape :", X_small.shape)
subsample = np.random.choice(len(X_scaled), 10000, replace=False)
X_sub = X_scaled[subsample]
birch_scaled = []
threshold_values = [0.2, 0.5, 1.0, 1.5]
for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_sub)
    n_clusters = len(set(labels))
    #X_scaled = X_scaled.astype("float32")
    if 1 < n_clusters < len(X_sub) :  #and len(set(labels[mask])) > 1
        sil, db, ch = compute_metrics(X_sub, labels)
        birch_scaled.append({
            "algorithm": "BIRCH",
            "preprocessing": "Scaled",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 30.654717206954956 seconds
BIRCH runtime: 30.6547 seconds


In [4]:
#Optics on Scaled Data
start_time = time.time()
optics_scaled = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_scaled)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_scaled, labels)
        optics_scaled.append({
            "algorithm": "OPTICS",
            "preprocessing": "Scaled",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 12188.571017503738 seconds
Optics runtime: 12188.5710 seconds


In [23]:
import csv


pain_results_scaled = (birch_scaled+spec_scaled+agg_scaled)

keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/pain_data/pain_scaled.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(pain_results_scaled)